In [1]:
import os   
%pwd

'c:\\Users\\danm\\Documents\\DLOps_Project\\research'

In [2]:
os.chdir("../")
%pwd

'c:\\Users\\danm\\Documents\\DLOps_Project'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path


In [4]:
from src.dlProject_energy_demand_forcasting.constants import *
from src.dlProject_energy_demand_forcasting.utils.utils import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config

In [6]:
import os 
import urllib.request as request
import zipfile
from src.dlProject_energy_demand_forcasting.utils.logger import logging
from src.dlProject_energy_demand_forcasting.utils.utils import get_size

In [7]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_url,
                filename=self.config.local_data_file
            )
            logging.info(f"{filename} downloaded with the following info: \n{headers}")
        else:
            logging.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")
        if not zipfile.is_zipfile(self.config.local_data_file):
            with open(self.config.local_data_file, 'rb') as f:
                preview = f.read(300)
            raise ValueError(
                f"Downloaded file is not a valid zip.\n"
                f"File size: {os.path.getsize(self.config.local_data_file)} bytes\n"
                f"Preview: {preview}"
            )
    def extract_zip_file(self):
        
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [8]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e 

[ 2026-04-21 15:07:57,274 ] 31 root - INFO - yaml file: config\config.yaml loaded successfully
[ 2026-04-21 15:07:57,276 ] 31 root - INFO - yaml file: params.yaml loaded successfully
[ 2026-04-21 15:07:57,279 ] 31 root - INFO - yaml file: schema.yaml loaded successfully
[ 2026-04-21 15:07:57,280 ] 52 root - INFO - created directory at: artifacts
[ 2026-04-21 15:07:57,281 ] 52 root - INFO - created directory at: artifacts/data_ingestion
[ 2026-04-21 15:07:57,282 ] 13 root - INFO - File already exists of size: ~ 64 KB


ValueError: Downloaded file is not a valid zip.
File size: 65546 bytes
Preview: b'<!DOCTYPE html><html dir="ltr"><head><script nonce="0TLaLK9rE2rnuL9jT7L8Ag"> window[\'_DRIVE_VIEWER_ctiming\']={}; </script><meta name="google" content="notranslate"><meta http-equiv="X-UA-Compatible" content="IE=edge;"><style nonce="nVzakGkzGHTqGmSwkjGhTw">@font-face{font-family:\'Roboto\';font-style:i'